# HardMine-DMS trên Kaggle (GPU)

Notebook này chạy trong tài khoản Kaggle của bạn. **Không cần đưa API key cho ai.**

Trước khi chạy, ở thanh bên phải: **Settings → Accelerator → GPU T4 x2** (hoặc P100) và **Internet → On**.

Các bước:
1. Kiểm tra GPU, cài repo, chạy test.
2. Chạy demo trên dữ liệu tổng hợp.
3. Trích embedding bằng GPU cho video của bạn (đo thông lượng trước, rồi chạy toàn bộ).
4. Kiểm tra khử danh tính tài xế trên embedding thật.
5. Lưu kết quả để tải về hoặc đưa vào repo.

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, 'CUDA:', torch.cuda.is_available())

In [ ]:
!git clone -q https://github.com/luuthilananh/Hardmine_DMS.git /kaggle/working/Hardmine_DMS
%cd /kaggle/working/Hardmine_DMS
# torch, transformers, opencv đã có sẵn trên Kaggle; chỉ cài gói của repo
!pip install -q -e ".[dev]"
!python tests/run_tests.py | tail -3

## 2. Demo trên dữ liệu tổng hợp
Chứng minh pipeline chạy đầu-cuối. Kết quả này **không** phải hiệu năng trên dữ liệu thật.

In [ ]:
!python scripts/demo_synthetic.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/synthetic_demo/summary.md').read()))
display(Image('results/synthetic_demo/figures/label_efficiency.png'))
display(Image('results/synthetic_demo/figures/event_mining.png'))

## 3. Trích embedding bằng GPU

Thêm video của bạn qua **Add Input → Datasets** (tạo dataset **riêng tư**). Cấu trúc mong đợi:

```
/kaggle/input/<ten-dataset>/<tai_xe_1>/<phien_1>.mp4
/kaggle/input/<ten-dataset>/<tai_xe_2>/<phien_2>.mp4
```

Tên thư mục cha được dùng làm ID tài xế (cần cho khử danh tính và chia split theo người).

**Lưu ý giấy phép:** Drive&Act, DMD và các dataset buồn ngủ có điều khoản riêng. Không đăng chúng thành dataset công khai trên Kaggle; chỉ tải lên dạng riêng tư nếu điều khoản cho phép, nếu không hãy dùng video tự quay.

In [ ]:
VIDEO_DIR = '/kaggle/input/YOUR-DATASET'   # <- sửa tên dataset
PRESET = 'videomae_base'                    # hoặc 'vjepa2_vitl' (nặng hơn), 'siglip2_base' (ảnh)
OUT = f'/kaggle/working/features/{PRESET}'

# Bước 3a: đo thông lượng trên 200 cửa sổ trước
!python scripts/extract_features.py --videos $VIDEO_DIR --out {OUT}_bench --preset $PRESET --batch-size 8 --limit 200

In [ ]:
# Bước 3b: chạy toàn bộ (resume được nếu phiên bị ngắt: chạy lại đúng lệnh này)
!python scripts/extract_features.py --videos $VIDEO_DIR --out $OUT --preset $PRESET --batch-size 8

## 4. Kiểm tra khử danh tính tài xế trên embedding thật
Kỳ vọng (cổng G1): độ chính xác đoán ID tài xế giảm rõ khi tăng r. Cần ít nhất 3 tài xế.

In [ ]:
import pandas as pd, numpy as np
from hardmine.features.extract import load
from hardmine.features.deidentify import DriverProjector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

ids, z = load(OUT)
win = pd.read_csv(f'{OUT}/windows.csv').set_index('window_id').loc[ids]
drivers = win['driver_id'].to_numpy()
n_drv = len(np.unique(drivers))
print(z.shape, 'tài xế:', n_drv)

def acc(x, y):
    return cross_val_score(LogisticRegression(max_iter=2000), x, y, cv=3).mean()

rows = [{'rank': 0, 'driver_acc': acc(z, drivers)}]
for r in sorted({2, 4, 8, n_drv - 1}):
    if 0 < r < n_drv:
        zt = DriverProjector(rank=r).fit(z, drivers).transform(z)
        rows.append({'rank': r, 'driver_acc': acc(zt, drivers)})
pd.DataFrame(rows)

## 5. Lưu kết quả

Thư mục `/kaggle/working` được giữ lại sau khi **Save Version**; tải về từ tab Output.
Không commit embedding của dataset có điều khoản lên GitHub.

In [ ]:
!cd /kaggle/working && zip -qr hardmine_outputs.zip features Hardmine_DMS/results && ls -lh hardmine_outputs.zip